<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-02-embeddings-and-vectors/lesson-2.4-alloydb-bigquery/practice/GCP_Capstone_2.4_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 2.4 — AlloyDB pgvector & BigQuery

8 exercises with solutions. Master pgvector operators, ScaNN indexing, VECTOR_SEARCH(), and the decision framework.

Runnable companion to the published practice lab. Each exercise below shows the objective and a complete solution. Cloud Shell / `gcloud` steps are `%%bash` cells; Python steps run in Colab after you authenticate and set your project.

---

## Exercise 1: pgvector Distance Operators  
**Difficulty:** Easy

Write SQL for cosine, L2, and inner product search using the three pgvector operators.

1. Write a cosine distance query with <=>
2. Write an L2 distance query with <->
3. Write an inner product query with <#>

**Solution:**

In [ ]:
-- Cosine distance (DEFAULT for text embeddings)
SELECT content, embedding <=> '[0.1,0.2,...]'::vector AS cosine_dist
FROM documents ORDER BY cosine_dist LIMIT 5;

-- L2 / Euclidean distance
SELECT content, embedding <-> '[0.1,0.2,...]'::vector AS l2_dist
FROM documents ORDER BY l2_dist LIMIT 5;

-- Negative inner product (requires normalized vectors)
SELECT content, embedding <#> '[0.1,0.2,...]'::vector AS neg_ip
FROM documents ORDER BY neg_ip LIMIT 5;

## Exercise 2: Compare 3 Index Types  
**Difficulty:** Easy

Write CREATE INDEX statements for IVFFlat, HNSW, and ScaNN on the same table.

1. Create IVFFlat index with 100 lists
2. Create HNSW index with m=16, ef_construction=64
3. Create ScaNN index with 100 leaves

**Solution:**

In [ ]:
-- IVFFlat (any PostgreSQL)
CREATE INDEX idx_ivf ON documents
USING ivfflat (embedding vector_cosine_ops)
WITH (lists = 100);

-- HNSW (any PostgreSQL)
CREATE INDEX idx_hnsw ON documents
USING hnsw (embedding vector_cosine_ops)
WITH (m = 16, ef_construction = 64);

-- ScaNN (AlloyDB only!)
CREATE EXTENSION IF NOT EXISTS alloydb_scann CASCADE;
CREATE INDEX idx_scann ON documents
USING scann (embedding cosine)
WITH (num_leaves = 100);
ANALYZE documents;

## Exercise 3: BigQuery VECTOR_SEARCH  
**Difficulty:** Easy

Write a complete VECTOR_SEARCH query with inline AI.GENERATE_EMBEDDING for the query.

1. Reference a base table with embeddings
2. Generate query embedding inline
3. Return top 5 results with cosine distance

**Solution:**

In [ ]:
SELECT base.title, base.id, distance
FROM VECTOR_SEARCH(
  TABLE `my_dataset.doc_embeddings`,
  'ml_generate_embedding_result',
  (SELECT ml_generate_embedding_result AS embedding
   FROM AI.GENERATE_EMBEDDING(
     MODEL `my_dataset.embed_model`,
     (SELECT 'What is vector search?' AS content)
   )),
  top_k => 5,
  distance_type => 'COSINE'
);

## Exercise 4: AlloyDB + Python Pipeline  
**Difficulty:** Medium

Write Python code to connect to AlloyDB, insert 5 documents with embeddings, and query.

1. Connect with psycopg2 + register_vector
2. Generate embeddings with google-genai
3. Insert 5 docs with vector column
4. Query with <=> cosine distance

**Solution:**

In [ ]:
# See Colab Cell 2 for the complete AlloyDB Python pattern
# Key: psycopg2 + register_vector + google-genai embed_content

## Exercise 5: Filtered SQL Vector Search  
**Difficulty:** Medium

Write a query combining WHERE + JOIN + vector ORDER BY in one statement.

1. Filter by category and date range
2. JOIN with authors table
3. Order by cosine distance to query vector

**Solution:**

In [ ]:
SELECT d.content, a.name AS author,
       d.embedding <=> embedding('text-embedding-005',
           'How does attention work?')::vector AS distance
FROM documents d
JOIN authors a ON d.author_id = a.id
WHERE d.category = 'ai_ml'
  AND d.created_at > '2025-01-01'
ORDER BY d.embedding <=> embedding('text-embedding-005',
    'How does attention work?')::vector
LIMIT 5;

## Exercise 6: BigQuery Batch Embedding  
**Difficulty:** Medium

Use AI.GENERATE_EMBEDDING to embed rows from a table, create an IVF index, and search.

1. Create embedding model with REMOTE CONNECTION
2. Generate embeddings with AI.GENERATE_EMBEDDING
3. Create IVF vector index
4. Run VECTOR_SEARCH query

**Solution:**

In [ ]:
-- See Colab Cell 3 for the complete BigQuery SQL workflow
-- Steps: CREATE MODEL -> AI.GENERATE_EMBEDDING -> CREATE VECTOR INDEX -> VECTOR_SEARCH

## Exercise 7: Decision Framework Quiz  
**Difficulty:** Challenge

For each scenario, pick the right vector DB and justify your choice.

1. 500-doc prototype, zero budget
2. 50K docs, real-time chatbot, needs JOINs
3. 10M products, nightly batch recommendations
4. 1K docs, need ACID transactions
5. Data already in BigQuery, occasional similarity queries

**Solution:**

## Exercise 8: DocuMind Architecture Document  
**Difficulty:** Challenge

Write a 1-page architecture document for DocuMind specifying the vector database strategy.

1. Describe the Firestore prototype phase
2. Plan the AlloyDB production migration
3. Define BigQuery analytics integration
4. Specify embedding model, dimensions, task types

**Solution:**